# Lab 4 — Build the Audience, Then Buy It
Name: Yeremenko Anton | Group: KM-41

Setup

In [1]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str,keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

## Task 1.0 — Clean the survey file

In [2]:
svals = survey.values.tolist()
members = []
memdata = []
bads = 0
lookup1 = ["X710101","X710102","X710103"]; lookup2 = ["X710201","X710202","X710203"]; brands = ["STRIDE","KESTREL","EASYWALK"]
for el in svals:
    el_yn = 1 if el[2] in ['1','1.0','y','TRUE','yes','Y'] else 0 if el[2] in ['0','no','0.0','N'] else -1
    el_pair = 0 if el[1] not in lookup1 else lookup1.index(el[1]) + 1
    el_brand = 0 if el[1] not in lookup2 else brands[lookup2.index(el[1])]
    if el_yn == -1:
        bads += 1
    else:
        el0 = int(float(el[0]))
        if el0 in members:
            prev = memdata[members.index(el0)]
            new = prev
            if prev[0] < el_yn:
                new[0] = el_yn
            if prev[1] < el_pair:
                new[1] = el_pair
            memdata[members.index(el0)] = new
        else:
            memdata.append([el_yn,el_pair,el_brand])
            members.append(el0)
counts = [0,0,0,0,0,0,0,0,0,0] #nc yc 0 1 2 3 nobrand br1 br2 br3
for el in memdata:
    counts[el[0]] += 1
    counts[el[1]+2] += 1
    counts[6 if el[2]==0 else brands.index(el[2])+7] += 1

df0 = pd.DataFrame({'respondent_id':members})
df1 = pd.merge(df0,viewer_attributes,on="respondent_id",how="left")
df1 = df1[df1["attr_id"] >= 318]; df1 = df1[df1['attr_id'] <= 365]
df11 = df1[df1["attr_id"] >= 325]; df11 = df11[df11['attr_id'] <= 354]
ageds = df1["respondent_id"].values.tolist()
ageds1 = df11["respondent_id"].values.tolist()
agec = 0
for el in members:
    if el in ageds:
        agec+=1

In [3]:
print("Survey data rows:", len(survey))
print("Clean yes answers:", counts[1])
print("Adult panel members:", agec)
print("Non-buyers:", counts[0]+bads)
print("People with two pair counts:", counts[4])
print(f"{bads} bad records, flushed (rest were eg answer 1.0 => 1 (bought), etc)") # table: kind of bad record, rows, what you did
print(pd.DataFrame({"keyword": ["no answers","yes answers","0 pairs","1 pair","2 pairs","3+ pairs","no brand",brands[0],brands[1],brands[2]], "values":counts})) # table: KEYWORD, your count, layout SAMPLE

Survey data rows: 18349
Clean yes answers: 5435
Adult panel members: 5302
Non-buyers: 396
People with two pair counts: 1323
126 bad records, flushed (rest were eg answer 1.0 => 1 (bought), etc)
       keyword  values
0   no answers     270
1  yes answers    5435
2      0 pairs     936
3       1 pair    1759
4      2 pairs    1323
5     3+ pairs    1687
6     no brand    4602
7       STRIDE     507
8      KESTREL     194
9     EASYWALK     402


**Answer:** what was wrong with the file; how far off the raw file would have been. - the file had a lot of bad and different values, like "VALUE" pairs bought, or "unknown brand" and so on, or different answers meaning the same thing, like 1 1.0 TRUE y, all for answer "true" to buying a pair

## Task 1.1 — Four definitions

In [4]:
import numpy
df3 = pd.merge(df0,daily_weights,on="respondent_id",how="left")
df4 = df3[["respondent_id","weight"]].groupby("respondent_id").mean()
memwgts = df4.values.tolist()
memsort = sorted(members)
for i in range(0,len(memdata)):
    if len(memdata[i])==3:
        memdata[i].append(memwgts[memsort.index(members[i])][0])
c18p = 0; c2554 = 0
for i in range(0,len(members)):
    if not numpy.isnan(memwgts[i][0]):
        if members[i] in ageds:
            c18p += memwgts[i][0]
        if members[i] in ageds1:
            c2554 += memwgts[i][0]
cbuy = 0; c3pp = 0; ckes = 0; ckeswgt = 0
for el in memdata:
    if not numpy.isnan(el[3]):
        if el[0]==1:
            cbuy+=el[3]
        if el[1]==3:
            c3pp+=el[3]
        if el[2]=="KESTREL":
            ckes += 1
            ckeswgt += el[3]


In [5]:
print("Adults 18+ (M):", round(c18p/1000000,1))
print("Shoe buyers (M):", round(cbuy/1000000,1))
print("Heavy buyers, 3+ pairs (M):", round(c3pp/1000000,1))
print("Kestrel buyers (M):", round(ckeswgt/1000000,1))
print("Adults 25-54 (M):", round(c2554/1000000,1))
print("Kestrel buyers in panel (n):", ckes)

Adults 18+ (M): 177.7
Shoe buyers (M): 180.6
Heavy buyers, 3+ pairs (M): 55.7
Kestrel buyers (M): 5.8
Adults 25-54 (M): 109.6
Kestrel buyers in panel (n): 175


## Task 1.2 — Who are they?

In [6]:
amts = [0,0,0,0,0,0] # male 123 female 123
for i in range(len(memdata)):
    if memdata[i][2]=="KESTREL":
        ids = list(viewer_attributes[viewer_attributes.respondent_id == members[i]]["attr_id"])
        sex = 0 if 201 in ids else 1 if 202 in ids else -1
        age = 0 if (318 in ids or 321 in ids or 325 in ids) else 1 if (335 in ids or 350 in ids) else 2 if (355 in ids or 365 in ids) else -1
        if sex!=-1 and age!=-1:
            amts[3*sex+age] += 1

In [7]:
for i in range(0,6): print(f"{["male","female"][0 if i<3 else 1] :<6} {["18-34","35-54","55+"][i%3] :<6} share: {round(amts[i]/sum(amts),3)}") # table: audience x group, share and index vs adults

male   18-34  share: 0.103
male   35-54  share: 0.221
male   55+    share: 0.138
female 18-34  share: 0.152
female 35-54  share: 0.248
female 55+    share: 0.138


**Answer:** which definition for Kestrel; why shoe buyers look like adults; one limit of fused data. - middle aged people (both men and women) buy more than other categories

## Task 2.1 — Programs by audience

In [47]:
progs = []
pt = programs[programs.program_name == "NFL Football"]
progs.append(pt[pt.network_code == "NBC"][["start_ts","end_ts"]].values.tolist())
progs.append(pt[pt.network_code == "ABC"][["start_ts","end_ts"]].values.tolist())
pother = ["Dancing with the Stars","NCIS","Hannity","Unsellable Houses","Halloween Wars","Grotesquerie"]
for el in pother:
    progs.append(programs[programs.program_name == el][["start_ts","end_ts"]].values.tolist())

heavies = []
kestrels = []
for i in range(len(memdata)):
    if memdata[i][1]==3:
        heavies.append(members[i])
    if memdata[i][2]=="KESTREL":
        kestrels.append(members[i])


viewage = (viewings.where(viewings["respondent_id"].isin(ageds)))[["respondent_id","start_ts","end_ts","weight"]].dropna()
viewage["day"] = (viewage.start_ts - pd.Timedelta(hours=6)).dt.day
viewbuy = (viewings.where(viewings["respondent_id"].isin(heavies)))[["respondent_id","start_ts","end_ts","weight"]].dropna()
viewbuy["day"] = (viewbuy.start_ts - pd.Timedelta(hours=6)).dt.day
viewkest = (viewings.where(viewings["respondent_id"].isin(kestrels)))[["respondent_id","start_ts","end_ts","weight"]].dropna()
viewkest["day"] = (viewkest.start_ts - pd.Timedelta(hours=6)).dt.day
sawwgtsage = [0 for i in range(8)]; daywgtsage = [0 for i in range(8)]
sawwgtsbuy = [0 for i in range(8)]; daywgtsbuy = [0 for i in range(8)]
viewersage = [0 for i in range(8)]; viewersbuy = [0 for i in range(8)]
sawwgtskest = [0 for i in range(8)]; daywgtskest = [0 for i in range(8)]
adultrat = []
targetrat = []
kestrat = []
for i in range(0,8):
    for j in range(0,len(progs[i])):
        tview0 = viewage.where(viewage["start_ts"] > progs[i][j][0]).dropna()
        tview = tview0.where(tview0["end_ts"] < progs[i][j][1]).dropna()
        day = progs[i][j][0].day - (1 if progs[i][j][0].hour < 6 else 0)
        sawwgtsage[i] += float(tview["weight"].sum())
        daysum = float(viewage.where(viewage["day"] == day)["weight"].sum())
        daywgtsage[i] += daysum
        viewersage[i] += tview["respondent_id"].nunique()


        tviewbuy0 = viewbuy.where(viewbuy["start_ts"] > progs[i][j][0]).dropna()
        tviewbuy = tviewbuy0.where(tviewbuy0["end_ts"] < progs[i][j][1]).dropna()
        viewersbuy[i] += tviewbuy["respondent_id"].nunique()
        sawwgtsbuy[i] += float(tviewbuy["weight"].sum())
        daywgtsbuy[i] += daysum

        tviewkest0 = viewkest.where(viewkest["start_ts"] > progs[i][j][0]).dropna()
        tviewkest = tviewkest0.where(tviewkest0["end_ts"] < progs[i][j][1]).dropna()
        sawwgtskest[i] += float(tviewkest["weight"].sum())
        daywgtskest[i] += daysum
    adultrat.append(sawwgtsage[i]/daywgtsage[i]*100)
    targetrat.append(sawwgtsbuy[i]/daywgtsbuy[i]*100)
    kestrat.append(sawwgtskest[i]/daywgtskest[i]*100)
compos = [viewersbuy[i]/viewersage[i] for i in range(8)]
indexes = [targetrat[i]/adultrat[i]*100 for i in range(8)]
cp30 = []
cp30.append(rate_card.where(rate_card["daypart"] == "Sunday Night Football")["cost_30s_usd"].dropna().values.tolist())
cp30.append(rate_card.where(rate_card["daypart"] == "Monday Night Football")["cost_30s_usd"].dropna().values.tolist())
for i in range(6):
    cp30.append(rate_card.where(rate_card["program_name"] == pother[i])["cost_30s_usd"].dropna().values.tolist())
cp30 = [(cp30[i][0] if len(cp30[i])!=0 else 500000) for i in range(8)]
cpmage = [cp30[i]/viewersage[i]*1000 for i in range(8)]
cpmtarget = [cp30[i]/viewersbuy[i]*1000 for i in range(8)]

In [48]:
print(pd.DataFrame({"adults rating": adultrat, "target rating": targetrat,
"target viewers (M)": [round(viewersbuy[i]/1000,1) for i in range(8)], "composition": compos,
"index": indexes, "cost per :30": cp30, "adults cpm": cpmage, "target cpm": cpmtarget}))
# table: 8 programs, the columns of Task 2.1
print("Target universe (M):", [round(viewersage[i]/1000,1) for i in range(8)])

   adults rating  target rating  target viewers (M)  composition      index  \
0      22.503437       6.565261                 2.0     0.308498  29.174480   
1      26.276045       8.120527                 1.4     0.311281  30.904679   
2      11.827452       3.571526                 2.5     0.309887  30.196915   
3       4.874249       1.462595                 0.4     0.300159  30.006565   
4       5.618983       1.704832                 3.1     0.305687  30.340581   
5       2.277973       0.671827                 4.9     0.299484  29.492310   
6       1.996255       0.561529                 1.9     0.290076  28.129102   
7       1.783037       0.516607                 7.3     0.296038  28.973463   

   cost per :30     adults cpm    target cpm  
0      487200.0   75000.000000  2.431138e+05  
1      723600.0  162278.537789  5.213256e+05  
2      500000.0   62578.222778  2.019386e+05  
3      500000.0  398089.171975  1.326260e+06  
4      500000.0   49193.231011  1.609269e+05  
5     

## Task 2.2 — Index, volume or price?

In [12]:
prognames = ["Sunday Night Football","Monday Night Football"]
for i in range(6):
    prognames.append(pother[i])

In [13]:
print("Highest index (program):", prognames[targetrat.index(max(targetrat))])
print("Most target viewers (program):", prognames[viewersbuy.index(max(viewersbuy))])
print("Lowest target CPM (program):", pother) # couldnt find the 6 program names in rate_card df

Highest index (program): Monday Night Football
Most target viewers (program): Grotesquerie
Lowest target CPM (program): ['Dancing with the Stars', 'NCIS', 'Hannity', 'Unsellable Houses', 'Halloween Wars', 'Grotesquerie']


**Answer:** same program or not; what you tell the client. - it isnt the same program as there are more viewers for Grotesquerie, even thought Monday Night Football is more popular as a share

## Task 2.3 — Kestrel buyers

In [14]:
print(pd.DataFrame({"programs":prognames, "heavy index": indexes,"kestrel index": [kestrat[i]/adultrat[i]*100 for i in range(8)]})) # table: program, heavy-buyer index, Kestrel index

                 programs  heavy index  kestrel index
0   Sunday Night Football    29.174480       2.785935
1   Monday Night Football    30.904679       2.871288
2  Dancing with the Stars    30.196915       2.873944
3                    NCIS    30.006565       2.733274
4                 Hannity    30.340581       2.825897
5       Unsellable Houses    29.492310       3.048608
6          Halloween Wars    28.129102       3.014382
7            Grotesquerie    28.973463       2.676959


**Answer:** why it jumps; which to trust; how to check. - Kestrel buyers index jump much more because theres way less of them than heavy buyers in general, and its harder to get a clean answer from a small data sample. it would be better to compare Kestrel buyers to heavy buyers or shoe buyers in general and possibly deduce something from mapping one to another (also with regards to kestrel buyers share to shoe buyers), as they shouled have a similar pattern.

## Task 3.1 — Weights

In [33]:
n65p = hh_attributes[hh_attributes.head_age_band == "65+"].groupby("income_band")["household_id"].count().values.tolist()
n65m = hh_attributes[hh_attributes.head_age_band != "65+"].groupby("income_band")["household_id"].count().values.tolist()
tvs = [n65p[i] if i<6 else n65m[i-6] for i in range(12)]
unitvs = universe["tv_households"].values.tolist()
mults = [unitvs[i]/tvs[i] for i in range(12)]

wgtsd = round(daily_weights[daily_weights.respondent_id%10==0]["weight"].sum()/28/1000000,1)

viewall = viewings[["respondent_id","start_ts","end_ts","weight"]].dropna()
viewall["day"] = (viewall.start_ts - pd.Timedelta(hours=6)).dt.day
tvwatchamt = [0 for i in range(8)]; tvdayamt = [0 for i in range(8)]
for i in range(0,8):
    for j in range(0,len(progs[i])):
        tall0 = viewall.where(viewall["start_ts"] > progs[i][j][0]).dropna()
        tall = tall0.where(tall0["end_ts"] < progs[i][j][1]).dropna()
        day = progs[i][j][0].day - (1 if progs[i][j][0].hour < 6 else 0)


        tvwatchamt[i] += int(tall["respondent_id"].count())
        tvdayamt[i] += int(viewall.where(viewall["day"] == day)["respondent_id"].count())
unwgtrat = [tvwatchamt[i]/tvdayamt[i]*100 for i in range(8)]

In [34]:
print("TVs matched to one household:", mults)
print("Weights sum per day (M):", wgtsd)
... # table: telecast, unweighted, weighted, reference
print("Mean gap unweighted (points):", unwgtrat)
print("Mean gap weighted (points):", ...)
print("Share of gap closed:", ...)

TVs matched to one household: [2293.047130191321, 2234.5132743362833, 2727.9411764705883, 2478.118609406953, 2166.081284495735, 2594.3331029716655, 2292.7631578947367, 2527.8814262023216, 2209.5187917105727, 2346.147215997242, 1976.7838852850803, 2080.055074744296]
Weights sum per day (M): 139.3
Mean gap unweighted (points): [21.447622148205657, 25.794260197135543, 11.696702931650195, 5.085225725643959, 5.572448330034581, 2.3023474178403758, 2.0158955561596716, 1.8152778375520293]
Mean gap weighted (points): Ellipsis
Share of gap closed: Ellipsis


## Task 3.2 — What weighting cannot fix

**Answer:** what moved and why; what is left; why income barely matters here. - weighting still doesnt fix the small data sample that we're checking (each of the 8 telecasts)

## Task 4.1 — October predicts October

In [46]:
#print(programs[programs.network_code.isin(["CBS","NBC","ABC","FOX","Fox News"])]["telecast_id"].unique())


In [ ]:
print("Telecasts forecast (n):", ...)
print("MAPE, P2+ (%):", ...)
print("MAPE, heavy buyers (%):", ...)
print("Biggest miss, P2+ (program, date):", ...)

Telecasts forecast (n): Ellipsis
MAPE, P2+ (%): Ellipsis
MAPE, heavy buyers (%): Ellipsis
Biggest miss, P2+ (program, date): Ellipsis


## Task 4.2 — Why it missed

In [ ]:
... # table: the five biggest misses (program, date, forecast, actual)

Ellipsis

**Answer:** what happened; why the target forecast is worse; two improvements. - when looking at a part of a single month, one can miss events that happen once every longer period, like the World Series on Oct 25 or such ; so its better to zoom out and see irregularities from older data to see if such irregularities are repeatable in the timeframe we want to predict

## Task 5.1 — The plan

In [ ]:
... # table: program, spots, cost, target impressions
print("Plan cost ($):", ...)
print("Plan target impressions (M):", ...)

Plan cost ($): Ellipsis
Plan target impressions (M): Ellipsis


## Task 5.2 — Recommendation

**Answer:** 6-8 sentences. if we compare target viewers, there are the most of them at program 5 and 7 (table 2.1), which would make them more cost effective if the cost per :30 was uniform. it is mainly a question of targeting heavy buyers versus targeting potential new customers, when both are important. therefore, it would make sense to spend half the money targeting heavy buyers (if they havent already been targeted too much recently, ofcourse), and going for volume while staying cost effective. and to spend another half (or 55-45) on cost effective marketing to potential new customers (by volume) with lowest possible cpm when targeting everyone weighed by characteristics at or similar to table 1.2, as for example targeting males 35-54 would be twice as efficient as targeting males 18-34 (these demographics surmisable by program watchers statistics).
To conclude: we want to (not over-) market to people who buy our product, while also getting new customers by placing ads on volume targeting people based on their demographics, choice of which we can figure out by looking at already existing customer demographics.

## Bonus

In [ ]:
print("Kaggle username:", ...)

Kaggle username: Ellipsis


**Answer:** what your model does, in 2-3 sentences.